In [ ]:
from google.colab import drive
import os
import shutil
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2 as transforms
from torchvision import models
import datasets
import transformers
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, AutoModel, AutoModelForImageTextToText

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# shutil.rmtree("/content/data/PetImages")

In [ ]:
# drive_mount_path = '/content/drive'
# drive.mount(drive_mount_path)

# source_file_name = r"resnet_transfer_learning_selected_dataset.zip"
# zip_file_path = f"/content/drive/MyDrive/Colab Notebooks/{source_file_name}"

# !cp "{zip_file_path}" "/content/data"

# zip_file_path = f"/content/data/{source_file_name}"
zip_file_path = r"/content/cats_vs_dogs_image_classification_dataset.zip"

!unzip "{zip_file_path}" -d "/content/data"

unzip:  cannot find or open /content/cats_vs_dogs_image_classification_dataset.zip, /content/cats_vs_dogs_image_classification_dataset.zip.zip or /content/cats_vs_dogs_image_classification_dataset.zip.ZIP.


In [ ]:
# dataset_id = "Man1103/PetImages"
# petimg_dataset = datasets.load_dataset(dataset_id)

In [ ]:
def fetching_selected_data(
    cats_images_subpath: str,
    dogs_images_subpath: str,
    test_split: float,
    max_len: int = 1000000000
):
    def _get_all_image_files(parent_dir: str):
        all_files = []
        # Sort so train/test slices are deterministic
        for root, _, files in os.walk(parent_dir):
            for file in sorted(files):
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    all_files.append(os.path.join(root, file))
        return all_files

    all_cats = _get_all_image_files(cats_images_subpath)
    all_dogs = _get_all_image_files(dogs_images_subpath)

    total_len = min(len(all_cats), len(all_dogs), max_len)
    training_set_count = int(total_len * (1 - test_split))
    testing_set_count = total_len - training_set_count

    def _get_input_labels(first_count: int, max_count: int):
        last_count = first_count + max_count
        cats_slice = all_cats[first_count:last_count]
        dogs_slice = all_dogs[first_count:last_count]

        # Pair each file path directly with its known label (0 = cat, 1 = dog)
        labeled_files = [(path, 0) for path in cats_slice] + [(path, 1) for path in dogs_slice]

        X, y = [], []
        cats_counter, dogs_counter = 0, 0

        for idx, (file_path, label) in enumerate(labeled_files, 1):
            with Image.open(file_path) as img:
                X.append(img.convert("RGB"))
            y.append(label)

            if label == 0:
                cats_counter += 1
            else:
                dogs_counter += 1

            if idx % 1000 == 0:
                print(f"Processed {idx} images | Cats: {cats_counter} | Dogs: {dogs_counter}")

        return X, y, cats_counter, dogs_counter

    # Load Training Data
    X_train, y_train, train_cats, train_dogs = _get_input_labels(0, training_set_count)
    print(f"Final Training Data Counters | Cats: {train_cats} | Dogs: {train_dogs}")

    # Load Testing Data
    X_test, y_test, test_cats, test_dogs = _get_input_labels(training_set_count, testing_set_count)
    print(f"Final Testing Data Counters | Cats: {test_cats} | Dogs: {test_dogs}")

    return X_train, X_test, y_train, y_test


class MyDataset(Dataset):
    def __init__(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
        self.x_train_transform_pipeline = transforms.Compose([
            transforms.Resize((128, 128)), # Standardize size
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor()          # Converts PIL to Tensor & scales to [0,1]
        ])
        self.y_train_transform_pipeline = transforms.ToTensor()

    def __len__(self):
        return len(self.X_train)

    def __getitem__(self, idx):
        X_train_sample = self.X_train[idx] # Keep as PIL Image for the pipeline
        y_train_sample = self.y_train[idx]

        # Transform the image
        X_transformed = self.x_train_transform_pipeline(X_train_sample)

        # Convert label to tensor (LongTensor for CrossEntropyLoss)
        y_transformed = torch.tensor(y_train_sample, dtype=torch.long)

        return X_transformed, y_transformed


cats_images_subpath = r"/content/data/PetImages/Cat"
dogs_images_subpath = r"/content/data/PetImages/Dog"

(fetched_X_train, fetched_X_test,
 fetched_y_train, fetched_y_test) = fetching_selected_data(cats_images_subpath=cats_images_subpath,
                                                           dogs_images_subpath=dogs_images_subpath,
                                                           test_split=0.25, max_len=10000)

# Create Training Dataset and DataLoader
training_dataset = MyDataset(fetched_X_train, fetched_y_train)
training_dataloader = DataLoader(dataset=training_dataset, batch_size=128, shuffle=True, num_workers=2)

# Create Testing Dataset and DataLoader
testing_dataset = MyDataset(fetched_X_test, fetched_y_test)
testing_dataloader = DataLoader(dataset=testing_dataset, batch_size=256, shuffle=True, num_workers=1)

print(f"Size: Training | Dataset: {len(training_dataset)} | DataLoader: {len(training_dataloader)}")
print(f"Size: Testing | Dataset: {len(testing_dataset)} | DataLoader: {len(testing_dataloader)}")

In [ ]:
# Stage-1 of Fine-Tuning
resnet = models.resnet18(weights='DEFAULT')

for p in resnet.parameters():
    p.requires_grad = False

ft_stage = 2

if ft_stage == 1:
    # Stage-1 of Fine-Tuning
    resnet.fc = nn.Linear(in_features=512, out_features=2, bias=True)

    learning_rate = 1e-3

if ft_stage == 2:
    # Stage-2 of Fine-Tuning
    for layer in [resnet.layer3, resnet.layer4]:
        for p in layer.parameters():
            p.requires_grad = True
    for p in resnet.fc.parameters():
        p.requires_grad = True

    learning_rate = 1e-5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=learning_rate)

num_epochs = 15

resnet.to(device)

resnet.train()
for epoch in range(num_epochs):

    total_loss = torch.scalar_tensor(0)
    total_acc = torch.scalar_tensor(0)

    for idx, (X_train, y_train) in enumerate(training_dataloader):
        X_train = X_train.to(device)
        y_train = y_train.to(device)

        output = resnet(X_train)
        loss = criterion(output, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predictions = torch.argmax(output, dim=-1)
        correct = (predictions == y_train).sum().item()
        accuracy = (correct / y_train.size(0))

        total_loss += loss.item()
        total_acc += accuracy

    mean_epoch_loss = (total_loss / len(training_dataloader))
    mean_epoch_accuracy = (total_acc / len(training_dataloader))
    print(f"Epoch: {epoch}, Epoch_loss: {mean_epoch_loss.item()}, Epoch_accuracy: {mean_epoch_accuracy*100}")

Epoch: 0, Epoch_loss: 0.21818873286247253, Epoch_accuracy: 91.06400299072266
Epoch: 1, Epoch_loss: 0.10149579495191574, Epoch_accuracy: 96.08441162109375
Epoch: 2, Epoch_loss: 0.08008574694395065, Epoch_accuracy: 96.80064392089844
Epoch: 3, Epoch_loss: 0.06816750019788742, Epoch_accuracy: 97.42079162597656
Epoch: 4, Epoch_loss: 0.05382866784930229, Epoch_accuracy: 97.91326904296875
Epoch: 5, Epoch_loss: 0.04801495373249054, Epoch_accuracy: 98.20628356933594
Epoch: 6, Epoch_loss: 0.041997525840997696, Epoch_accuracy: 98.40574645996094
Epoch: 7, Epoch_loss: 0.037736065685749054, Epoch_accuracy: 98.68450927734375
Epoch: 8, Epoch_loss: 0.040333181619644165, Epoch_accuracy: 98.52510070800781
Epoch: 9, Epoch_loss: 0.029605822637677193, Epoch_accuracy: 98.99754333496094
Epoch: 10, Epoch_loss: 0.026387780904769897, Epoch_accuracy: 98.98667907714844
Epoch: 11, Epoch_loss: 0.030813125893473625, Epoch_accuracy: 98.77633666992188
Epoch: 12, Epoch_loss: 0.027763042598962784, Epoch_accuracy: 98.9983

In [ ]:
resnet.eval()
with torch.no_grad():
    test_loss = 0.0
    total_correct = 0
    total_samples = 0
    preds, labels = torch.Tensor([]).to(device), torch.Tensor([]).to(device)

    for idx, (X_test, y_test) in enumerate(testing_dataloader):
        X_test = X_test.to(device)
        y_test = y_test.to(device)

        # 1. Forward pass
        output = resnet(X_test)

        # 2. Compute the actual test loss for this batch
        # (Make sure 'criterion' matches your loss function name, e.g., nn.CrossEntropyLoss())
        batch_loss = criterion(output, y_test)
        test_loss += batch_loss.item()

        # 3. Track correct predictions
        predictions = torch.argmax(output, dim=-1)
        total_correct += (predictions == y_test).sum().item()
        total_samples += y_test.size(0)

        preds = torch.cat([preds, predictions])
        labels = torch.cat([labels, y_test])

    # Calculate final averages
    mean_test_loss = test_loss / len(testing_dataloader)
    final_test_accuracy = (total_correct / total_samples) * 100

    print(f"Test Loss: {mean_test_loss:.4f}, Test Accuracy: {final_test_accuracy:.2f}%")

Test_loss: 0.04486095532774925, Test_accuracy: 92.3076923076923


In [ ]:
# 1. Move to CPU and convert to numpy
model_y_prob = preds.detach().cpu().numpy()
model_y_true = labels.detach().cpu().numpy()

# 2. Convert probabilities to hard binary labels (0 or 1)
model_y_pred = (model_y_prob >= 0.5).astype(int)

# 3. Calculate metrics
accuracy = accuracy_score(y_true=model_y_true, y_pred=model_y_pred)
precision = precision_score(y_true=model_y_true, y_pred=model_y_pred)
recall = recall_score(y_true=model_y_true, y_pred=model_y_pred)
f1 = f1_score(y_true=model_y_true, y_pred=model_y_pred)

print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1}")

In [ ]:
cm = confusion_matrix(y_true=model_y_true, y_pred=model_y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"tn: {tn}, fp: {fp}, fn: {fn}, tp: {tp}")

In [ ]:
print(classification_report(y_true=model_y_true, y_pred=model_y_pred))

In [ ]:
layerwise_param_count = {"layer_name": [], "param_count": [], "param_type": []}
for name, p in resnet.named_parameters():
    name_parts = name.split(".")
    layer_name = "_".join(name_parts[:-1])
    param_count = p.numel()
    param_type = "weight" if "weight" in name_parts[-1] else "bias" if "bias" in name_parts[-1] else "undefined"

    layerwise_param_count["layer_name"].append(layer_name)
    layerwise_param_count["param_count"].append(param_count)
    layerwise_param_count["param_type"].append(param_type)

model_capacity = sum(layerwise_param_count["param_count"])
print(f"Model capacity : {model_capacity} or {model_capacity/1000000} million parameters")

plt.figure(figsize=(20, 6))
sns.barplot(x=layerwise_param_count["layer_name"], y=layerwise_param_count["param_count"], hue=layerwise_param_count["param_type"], palette="tab10")
plt.xticks(rotation=90)
plt.yscale("log")
plt.title("Layerwise wise parameter count")
plt.show()

In [ ]:
# class BasicCNNModel(nn.Module):
#     def __init__(self, *args, **kwards):
#         super().__init__()

#         self.cnn_model = nn.Sequential(
#             nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1),
#             # nn.BatchNorm2d(num_features=32),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2),
#             nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
#             # nn.BatchNorm2d(num_features=64),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2),

#             nn.Flatten(),

#             nn.Linear(64 * 32 * 32, 16),
#             nn.ReLU(),
#             nn.Linear(16, 8),
#             nn.ReLU(),
#             nn.Linear(8, 2),
#             # nn.Sigmoid(),
#         )

#     def forward(self, X: torch.Tensor):
#         return self.cnn_model(X)

# model = BasicCNNModel()
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# num_epochs = 5

# for epoch in range(num_epochs):

#     epoch_loss = torch.scalar_tensor(0)
#     epoch_accuracy = torch.scalar_tensor(0)

#     for idx, (X_train, y_train) in enumerate(training_dataloader):
#         output = model(X_train)
#         loss = criterion(output, y_train)

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         _, predicted = torch.max(output, 1)
#         correct = (predicted == y_train).sum().item()
#         accuracy = (correct / y_train.size(0))*100

    #     epoch_loss = epoch_loss + loss
    #     epoch_accuracy = epoch_accuracy + accuracy

#     print(epoch, epoch_loss.item(), epoch_accuracy.item())

In [ ]:
from huggingface_hub import login, HfApi
from safetensors.torch import save_model, save_file

login()

api = HfApi()

In [ ]:
repo_name = "Man1103/PetImages"
api.create_repo(repo_name, repo_type="dataset", exist_ok=True)

# import os
# import shutil

# part_num = 0
# parent_dir = r"/content/data/PetImages"
# target_dir = r"/content/data/petimages"

# if os.path.exists(target_dir):
#     shutil.rmtree(target_dir)
# os.makedirs(target_dir, exist_ok=True)

# for child in os.listdir(parent_dir):
#     child_full_path = os.path.join(parent_dir, child)

#     if not os.path.isdir(child_full_path):
#         continue

#     target_sub_path = os.path.join(target_dir, child.lower())
#     os.makedirs(target_sub_path, exist_ok=True)

#     counter = 0
#     target_part_path = None

#     for grandchild in os.listdir(child_full_path):
#         # Create a new subfolder immediately at 0, 999, 1998, etc.
#         if counter % 999 == 0:
#             target_part_path = os.path.join(target_sub_path, f"part_{part_num}")
#             os.makedirs(target_part_path, exist_ok=True)
#             part_num += 1

#         grandchild_full_path = os.path.join(child_full_path, grandchild)

#         try:
#             shutil.move(grandchild_full_path, target_part_path)
#         except Exception as e:
#             print(f"Error moving {grandchild}: {e}")

#         counter += 1

# for dir_id in os.listdir(r"/content/data/petimages/dog"):
#     print(len(os.listdir(os.path.join(r"/content/data/petimages/dog", dir_id))), dir_id)

# dataset = {
#     "inputs": training_dataset[:][0],
#     "targets": training_dataset[:][1],
# }
# save_file(dataset, "/content/train.safetensors")
# dataset = {
#     "inputs": testing_dataset[:][0],
#     "targets": testing_dataset[:][1],
# }
# save_file(dataset, "/content/test.safetensors")

torch.save(training_dataset, "/content/train.pt")
torch.save(testing_dataset, "/content/test.pt")

api.upload_file(
    path_or_fileobj=r"/content/train.pt",
    path_in_repo="train.pt",
    repo_id=repo_name,
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj=r"/content/test.pt",
    path_in_repo="test.pt",
    repo_id=repo_name,
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj=r"/content/data/PetImages",
    path_in_repo="data",
    repo_id=repo_name,
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj=r"/content/cats_vs_dogs_image_classification_dataset.zip",
    path_in_repo="cats_vs_dogs_images.zip",
    repo_id=repo_name,
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj=r"/content/data/preprocessing.py",
    path_in_repo="preprocessing.py",
    repo_id=repo_name,
    repo_type="dataset",
)

NameError: name 'torch' is not defined

In [ ]:
repo_name = "Man1103/ResNet18-FT-PetImages-11.7M"
api.create_repo(repo_name, repo_type="model", exist_ok=True)

save_model(resnet, "/content/model.safetensors")

api.upload_file(
    path_or_fileobj=r"/content/model.safetensors",
    path_in_repo="model.safetensors",
    repo_id=repo_name,
    repo_type="model",
)

api.upload_file(
    path_or_fileobj=r"/content/preprocessing.py",
    path_in_repo="preprocessing.py",
    repo_id=repo_name,
    repo_type="model",
)

api.upload_file(
    path_or_fileobj=r"/content/train.py",
    path_in_repo="train.py",
    repo_id=repo_name,
    repo_type="model",
)

NameError: name 'resnet' is not defined